In [304]:
import mlflow
import pandas as pd
import torch
from pyarrow import Tensor
from sympy import tensor
from torch import nn
import numpy as np
from hyperopt import STATUS_OK,Trials,fmin,hp,tpe
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

from mlflow.models import infer_signature
from torch.nn import Softmax

In [305]:
data = pd.read_csv("https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv",
                   sep = ";")

In [306]:
y = data[["quality"]]
X = data.drop("quality", axis=1)

In [307]:
X = torch.tensor(X.values, dtype=torch.float32)
y = torch.tensor(y.values, dtype=torch.float32)

In [308]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [309]:
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

torch.Size([3918, 11]) torch.Size([3918, 1]) torch.Size([980, 11]) torch.Size([980, 1])


In [310]:
class LogisticRegression(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.Linear(11, 30)
        self.a1 = nn.ReLU()
        self.b1 = nn.BatchNorm1d(30)

        self.l2 = nn.Linear(30, 50)
        self.a2 = nn.ReLU()
        self.b2 = nn.BatchNorm1d(50)

        self.l3 = nn.Linear(50,1)

    def forward(self, x):
        x = self.b1(self.a1(self.l1(x)))
        x = self.b2(self.a2(self.l2(x)))
        return self.l3(x)

In [311]:
lr_nn = LogisticRegression()

In [312]:
optim = torch.optim.Adam(lr_nn.parameters(),lr=0.0001)

In [313]:
loss_fn = nn.CrossEntropyLoss()

In [314]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [315]:
mlflow.set_tracking_uri("sqlite:////Users/benjaminbrooke/PycharmProjects/MLOps/Complete_MLOps_Bootcamp_With_10+_End_To_End_ML_Projects/3.MLFlow/mlflow.db")

#mlflow ui --backend-store-uri sqlite:////Users/benjaminbrooke/PycharmProjects/MLOps/Complete_MLOps_Bootcamp_With_10+_End_To_End_ML_Projects/3.MLFlow/mlflow.db

mlflow.set_experiment("Deep Learning Logistic Regression")

<Experiment: artifact_location='/Users/benjaminbrooke/PycharmProjects/MLOps/Complete_MLOps_Bootcamp_With_10+_End_To_End_ML_Projects/3.MLFlow/mlruns/6', creation_time=1776532722150, experiment_id='6', last_update_time=1776532722150, lifecycle_stage='active', name='Deep Learning Logistic Regression', tags={}, trace_location=None, workspace='default'>

In [316]:
from torch.utils.data import DataLoader, TensorDataset

dataset = TensorDataset(X_train, y_train)

loader = DataLoader(dataset, batch_size=32, shuffle=True)

In [317]:
with mlflow.start_run():

    for epoch in range(400):

        epoch_loss = 0

        for X, y in loader:

            optim.zero_grad()

            predict = lr_nn(X)

            loss = loss_fn(predict, y)

            loss.backward()

            optim.step()

            epoch_loss += loss.item()

        epoch_loss /= len(loader)

        y_pred_list = []
        y_true_list = []

        for X, y in loader:
            preds = lr(X).detach()
            y_pred_list.append(preds)
            y_true_list.append(y)

        y_pred = torch.cat(y_pred_list).numpy()
        y_true = torch.cat(y_true_list).numpy()



        acc = accuracy_score(y_true, y_pred.argmax(axis=1))

        print(acc)

        precision = precision_score(y_true, y_pred.argmax(axis=1), average="macro", zero_division=0)
        recall = recall_score(y_true, y_pred.argmax(axis=1), average="macro", zero_division=0)
        f1 = f1_score(y_true, y_pred.argmax(axis=1), average="macro", zero_division=0)


"""        mlflow.log_metric("loss", epoch_loss, step=epoch)
        mlflow.log_metric("accuracy", acc, step=epoch)
        mlflow.log_metric("precision", precision, step=epoch)
        mlflow.log_metric("recall", recall, step=epoch)
        mlflow.log_metric("f1", f1, step=epoch)"""

RuntimeError: 0D or 1D target tensor expected, multi-target not supported